In [ ]:
!pip install librosa tensorflow scikit-learn

In [ ]:
import os
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
emotion_map = {
    "03": "happy",
    "04": "sad",
    "05": "angry"
}

data = []
labels = []

dataset_path = "/content/drive/MyDrive/audio_speech_actors_01-24"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            emotion_code = file.split("-")[2]
            if emotion_code in emotion_map:
                file_path = os.path.join(root, file)
                data.append(file_path)
                labels.append(emotion_map[emotion_code])

print("Total samples:", len(data))

Total samples: 584


In [ ]:
def extract_features(file_path):
    audio, sample_rate = librosa.load(file_path, duration=3, offset=0.5)

    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)

    max_len = 130

    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, pad_width=((0,0),(0,pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :max_len]

    return mfcc


In [ ]:
features = []

for file in data:
    features.append(extract_features(file))

X = np.array(features)
print("Shape of X:", X.shape)

Shape of X: (584, 40, 130)


In [ ]:
X = X[..., np.newaxis]
print("Shape after adding channel:", X.shape)

Shape after adding channel: (584, 40, 130, 1)


In [ ]:
le = LabelEncoder()
y = le.fit_transform(labels)
y = to_categorical(y)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

Feature shape: (584, 40, 130, 1)
Label shape: (584, 3)


In [ ]:
print(le.classes_)

['angry' 'happy' 'sad']


In [ ]:
X = X / np.max(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)

(467, 40, 130, 1)


In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(40,130,1)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(3, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
history = model.fit(X_train,y_train,epochs=50,batch_size=16,validation_data=(X_test, y_test))

Epoch 1/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 126ms/step - accuracy: 0.4600 - loss: 1.0821 - val_accuracy: 0.4615 - val_loss: 1.0228
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 158ms/step - accuracy: 0.5633 - loss: 0.9359 - val_accuracy: 0.7009 - val_loss: 0.8249
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.6654 - loss: 0.7577 - val_accuracy: 0.6923 - val_loss: 0.7445
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 114ms/step - accuracy: 0.7574 - loss: 0.5727 - val_accuracy: 0.7863 - val_loss: 0.5755
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 152ms/step - accuracy: 0.8331 - loss: 0.4825 - val_accuracy: 0.7778 - val_loss: 0.6321
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 125ms/step - accuracy: 0.8716 - loss: 0.3515 - val_accuracy: 0.7863 - val_loss: 0.5506
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.8745 - loss: 0.3509 - val_accuracy: 0.8291 - val_loss: 0.4820
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.9214 - loss: 0.2396 - val_accuracy: 0.

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8135 - loss: 1.0316
Test Accuracy: 0.8461538553237915


In [ ]:
model.save("SER_model.h5")

In [ ]:
from google.colab import files
files.download("SER_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>